In [ ]:
import json

import pandas as pd
import numpy as np
import requests

import boto3

1.テストデータ読み込み

In [ ]:
df = pd.read_csv("../data/X_test.csv")

2.ローカルでコンテナの動作確認

In [ ]:
payload = {
    "inputs": df.head(2).replace({np.nan: None}).to_dict("records")
}


In [ ]:
url = "http://localhost:8080/invocations"
headers = {"Content-Type": "application/json"}

In [ ]:
response = requests.post(url, 
                         headers=headers, 
                         data=json.dumps(payload))

In [ ]:
if response.status_code == 200:
    print("推論成功")
    print("レスポンス:", response.json())
else:
    print(f"推論エラー。ステータスコード: {response.status_code}")
    print("エラー内容:", response.text)

3.SageMakerのエンドポイントでの動作確認

In [ ]:
ENDPOINT_NAME = "demand-forecast"
REGION_NAME = "ap-northeast-1"         

In [ ]:
sagemaker_runtime = boto3.client(
    "sagemaker-runtime", 
    region_name=REGION_NAME
)

In [ ]:
payload = {
    "inputs": df.head(2).replace({np.nan: None}).to_dict("records")
}

In [ ]:
try:
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",            # inference.py の input_fn で受け取るタイプ
        Accept="application/json",                 # inference.py の output_fn で返すタイプ
        Body=json.dumps(payload)                    # JSON文字列に変換してBodyに渡す
    )

    
    # レスポンスの "Body" は StreamingBody オブジェクト（ストリーム形式）。
    # そのため.read() で中身を読み込み、文字列（utf-8）にデコードする。
    response_body = response["Body"].read().decode("utf-8")
    result = json.loads(response_body)

    print("レスポンス中身:", result)

except Exception as e:
    print(f"\nエラー: {e}")